# 02 · Metrics & Efficiency Analysis

Publication figures and tables for the 5-phase study, built **only** from the consolidated files in
`<pipeline>/summary/` (written by `yolo26_seg/build_final_report.py`) plus the raw latency samples in
`phase5_test/efficiency/*.json`. No GPU is needed.

| Figure | Question it answers |
|---|---|
| 1 | Does accuracy (DSC, JSI) hold from cross-validation to the test set, and what does HPO add? |
| 2 | Is the HPO gain on the test set significant (paired, per image)? |
| 3 | How does accuracy scale with model size and compute? |
| 4 | Latency vs. FPS for FP32 and FP16 (network forward vs. full `predict()`). |
| 5 | How stable is the latency (median and P95)? |
| 6 | Memory footprint (VRAM, RAM). |
| 7 | Accuracy–latency trade-off for deployment. |

Figures are saved as PDF (vector, for LaTeX) and PNG under `<pipeline>/figures/`; LaTeX tables under `<pipeline>/tables/`.
Colours follow a CVD-validated categorical palette; every chart also has a legend and a table view.

In [ ]:
# ---- Configuration ----------------------------------------------------------
PIPELINE_NAME = "pipeline_final_v1"   # sub-directory of logs/ (or set $PIPELINE_DIR)
DEPLOY_VARIANT = "optimized"          # variant shown in the efficiency figures


import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Locate the repository and the pipeline outputs ------------------------
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def find_pipeline_dir(name: str) -> Path:
    """$PIPELINE_DIR, else the Docker mount, else <repo>/logs/<name>."""
    candidates = [os.environ.get("PIPELINE_DIR"), f"/workspace/logs/{name}", REPO_ROOT / "logs" / name]
    for c in candidates:
        if c and Path(c).is_dir():
            return Path(c)
    raise FileNotFoundError(f"Pipeline outputs not found; tried {candidates}. Set PIPELINE_DIR.")


def local_path(p) -> Path:
    """Map /workspace/... paths recorded inside Docker to this checkout when run on the host."""
    p = Path(p)
    if not p.exists() and str(p).startswith("/workspace/"):
        alt = REPO_ROOT / p.relative_to("/workspace")
        if alt.exists():
            return alt
    return p


PIPELINE_DIR = find_pipeline_dir(PIPELINE_NAME)
FIG_DIR = PIPELINE_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

# ---- Figure style (publication, light) -------------------------------------
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#ffffff"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "semibold", "axes.titlelocation": "left",
    "axes.labelcolor": INK_2, "axes.edgecolor": INK_2, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "legend.frameon": False, "legend.fontsize": 8, "lines.linewidth": 2, "lines.markersize": 7,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def save(fig, name: str) -> None:
    """Save a figure as vector PDF (for LaTeX) and 300-dpi PNG under <pipeline>/figures/."""
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=300)
    print(f"saved {FIG_DIR / name}.pdf|png")


print("pipeline :", PIPELINE_DIR)
print("figures  :", FIG_DIR)


MODELS = ["nano", "small", "medium", "large", "xlarge"]
SHORT = {"nano": "n", "small": "s", "medium": "m", "large": "l", "xlarge": "x"}
# Categorical colours: validated light-mode slots (all-pairs CVD ΔE ≥ 9). Colour follows the entity.
SERIES = {"cv": "#2a78d6", "baseline": "#eb6834", "optimized": "#1baf7a"}
SERIES_LABEL = {"cv": "Phase 2 · 5-fold CV (held-out folds)",
                "baseline": "Phase 5 · Test — Baseline", "optimized": "Phase 5 · Test — Optimized"}
PREC = {"fp32": "#2a78d6", "fp16": "#4a3aa7"}
MARKER = {"baseline": "s", "optimized": "o", "fp32": "o", "fp16": "^"}
TABLE_DIR = PIPELINE_DIR / "tables"
TABLE_DIR.mkdir(exist_ok=True)

In [ ]:
SUMMARY = PIPELINE_DIR / "summary"


def read_csv(name: str) -> pd.DataFrame:
    p = SUMMARY / f"{name}.csv"
    if not p.exists():
        print(f"[missing] {p} — run build_final_report.py")
        return pd.DataFrame()
    return pd.read_csv(p)


acc, eff, gain, cv_pix, cv_inst = (read_csv(n) for n in
                                   ("test_accuracy", "efficiency", "hpo_gain", "phase2_cv_pixel", "phase2_cv_baseline"))
report = json.loads((SUMMARY / "final_results.json").read_text()) if (SUMMARY / "final_results.json").exists() else {}

present = set(acc["model"]) if len(acc) else set()
MODELS = [m for m in MODELS if m in present] or MODELS
X = np.arange(len(MODELS))
acc32 = acc[acc.precision == "fp32"] if len(acc) else acc

print(f"models: {MODELS}")
for w in report.get("warnings", []):
    print("WARNING:", w)

## Figure 1 — DSC and JSI across phases

Per model: Phase 2 cross-validation (mean ± sample SD over the 5 held-out folds; Baseline protocol), and the test set
for the Baseline and the Optimized models (per-image mean with seeded bootstrap 95 % CI). A dot plot is used because
the values are close together; bars would have to start at zero and hide the differences.

In [ ]:
def series_stats(key: str, metric: str):
    """(mean, 2×N error) — CV: ± sample SD over folds; test: bootstrap 95 % CI."""
    if key == "cv":
        d = cv_pix.set_index("model").reindex(MODELS)
        return d[f"{metric}_mean"].to_numpy(), np.vstack([d[f"{metric}_std"].to_numpy()] * 2)
    d = acc32[acc32.variant == key].set_index("model").reindex(MODELS)
    m = d[f"{metric}_mean"].to_numpy()
    return m, np.vstack([m - d[f"{metric}_ci95_low"].to_numpy(), d[f"{metric}_ci95_high"].to_numpy() - m])


fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9))
offsets = {"cv": -0.22, "baseline": 0.0, "optimized": 0.22}
for ax, metric, title in zip(axes, ("dsc", "jsi"), ("Dice similarity (DSC)", "Jaccard index (JSI)")):
    for key in ("cv", "baseline", "optimized"):
        if key == "cv" and cv_pix.empty:
            continue
        mean, err = series_stats(key, metric)
        ax.errorbar(X + offsets[key], mean, yerr=err, fmt=MARKER.get(key, "D"),
                    color=SERIES[key], ecolor=SERIES[key], elinewidth=1.2, capsize=0,
                    markeredgecolor=SURFACE, markeredgewidth=1.5, markersize=7, label=SERIES_LABEL[key])
    ax.set_xticks(X, [SHORT[m] for m in MODELS])
    ax.set_xlabel("YOLO26-seg size")
    ax.set_title(title)
    ax.set_ylabel(metric.upper())
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.08))
fig.tight_layout(rect=(0, 0.06, 1, 1))
save(fig, "fig1_dsc_jsi_across_phases")
plt.show()

table1 = pd.DataFrame({"model": MODELS})
if not cv_pix.empty:
    table1 = table1.merge(cv_pix[["model", "dsc_mean", "dsc_std", "jsi_mean", "jsi_std"]]
                          .rename(columns=lambda c: c if c == "model" else f"cv_{c}"), on="model", how="left")
for v in ("baseline", "optimized"):
    d = acc32[acc32.variant == v][["model", "dsc_mean", "dsc_ci95_low", "dsc_ci95_high", "jsi_mean"]]
    table1 = table1.merge(d.rename(columns=lambda c: c if c == "model" else f"test_{v}_{c}"), on="model", how="left")
table1.round(4)

## Figure 2 — HPO gain on the test set (paired)

Optimized − Baseline per image on the same test images: mean difference with paired bootstrap 95 % CI.
The label is the two-sided Wilcoxon signed-rank *p*-value. An interval that crosses zero means no reliable gain.

In [ ]:
def fmt_p(p: float) -> str:
    """APA-style p-value label."""
    if not np.isfinite(p):
        return "p = n/a"
    return "p<.001" if p < 0.001 else f"p={p:.3f}".replace("0.", ".")


if gain.empty:
    print("hpo_gain.csv missing — run Phase 5")
else:
    g = gain.set_index("model").reindex(MODELS)
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6), sharey=True)
    for ax, k, title in zip(axes, ("dsc", "jsi"), ("Δ DSC (Optimized − Baseline)", "Δ JSI (Optimized − Baseline)")):
        d = g[f"delta_{k}"].to_numpy()
        err = np.vstack([d - g[f"delta_{k}_ci95_low"].to_numpy(), g[f"delta_{k}_ci95_high"].to_numpy() - d])
        ax.axhline(0, color=INK_2, linewidth=0.8)
        ax.errorbar(X, d, yerr=err, fmt="o", color=SERIES["optimized"], elinewidth=1.2, capsize=0,
                    markeredgecolor=SURFACE, markeredgewidth=1.5, markersize=7)
        for xi, di, hi, p in zip(X, d, g[f"delta_{k}_ci95_high"], g[f"wilcoxon_p_{k}"]):
            ax.annotate(fmt_p(p), (xi, hi), xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=7, color=INK_2)
        lo, hi_ = ax.get_ylim()
        ax.set_ylim(min(lo, 0), hi_ + 0.18 * (hi_ - min(lo, 0)))  # headroom for the p labels
        ax.set_xlim(-0.6, len(MODELS) - 0.4)
        ax.set_xticks(X, [SHORT[m] for m in MODELS])
        ax.set_xlabel("YOLO26-seg size")
        ax.set_title(title)
    axes[0].set_ylabel("mean paired difference")
    fig.tight_layout()
    save(fig, "fig2_hpo_gain")
    plt.show()
    display(gain.set_index("model").reindex(MODELS)[
        ["n_images", "dsc_baseline", "dsc_optimized", "delta_dsc", "delta_dsc_ci95_low", "delta_dsc_ci95_high",
         "wilcoxon_p_dsc", "n_improved_dsc", "n_worse_dsc", "delta_jsi", "wilcoxon_p_jsi", "delta_map5095_m"]].round(4))

## Figure 3 — Accuracy vs. model size and compute

Test-set mask mAP@50-95 and DSC against the FP32 weight size (fused parameters × 4 bytes) and GFLOPs at 640×640.
Lines connect the sizes of each variant in order (n → x). The x-axes are logarithmic.

In [ ]:
cost = (eff[(eff.variant == DEPLOY_VARIANT) & (eff.precision == "fp32")]
        .set_index("model")[["size_mb_fp32_theoretical", "size_mb_disk", "gflops", "params_fused"]])
fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.2), sharex="col")
for row, (metric, ylab) in enumerate((("map5095_m", "mask mAP@50-95 (test)"), ("dsc_mean", "DSC (test)"))):
    for col, (cx, xlab) in enumerate((("size_mb_fp32_theoretical", "model size, FP32 weights (MB)"), ("gflops", "GFLOPs @ 640"))):
        ax = axes[row, col]
        for v in ("baseline", "optimized"):
            d = acc32[acc32.variant == v].set_index("model").reindex(MODELS).join(cost)
            ax.plot(d[cx], d[metric], color=SERIES[v], marker=MARKER[v], markeredgecolor=SURFACE,
                    markeredgewidth=1.5, label=SERIES_LABEL[v].split("— ")[1])
            if v == "optimized":
                for m, xv, yv in zip(MODELS, d[cx], d[metric]):
                    ax.annotate(SHORT[m], (xv, yv), xytext=(5, -10), textcoords="offset points", fontsize=8, color=INK_2)
        ax.set_xscale("log")
        ax.grid(True, axis="both")
        if row == 1:
            ax.set_xlabel(xlab)
        if col == 0:
            ax.set_ylabel(ylab)
axes[0, 0].legend(loc="lower right")
fig.tight_layout()
save(fig, "fig3_accuracy_vs_size")
plt.show()
t3 = acc32.pivot(index="model", columns="variant", values=["map5095_m", "dsc_mean"]).reindex(MODELS)
t3.columns = [f"{v} · {k}" for k, v in t3.columns]
t3.join(cost).round(4)

## Figure 4 — Latency vs. FPS (batch = 1)

Each point is one model at one precision. Filled markers: network forward pass (timed with `torch.cuda.Event`);
hollow markers: full `predict()` (pre-processing + inference + mask post-processing). FPS = 1000 / mean latency,
so points lie near the grey reference curve; what matters is how far along it each model sits and the
forward → end-to-end gap. Baseline and Optimized share the architecture, so only the deployed variant is shown.

In [ ]:
e = eff[eff.variant == DEPLOY_VARIANT]
fig, ax = plt.subplots(figsize=(5.2, 3.4))
lat = np.linspace(max(0.5, e[["fwd_median_ms", "e2e_median_ms"]].min().min() * 0.8),
                  e[["fwd_median_ms", "e2e_median_ms"]].max().max() * 1.15, 200)
ax.plot(lat, 1000 / lat, color=GRID, linewidth=1.5, zorder=0, label="FPS = 1000 / latency")
for prec in ("fp32", "fp16"):
    d = e[e.precision == prec].set_index("model").reindex(MODELS)
    ax.scatter(d.fwd_median_ms, d.fwd_fps, color=PREC[prec], marker=MARKER[prec], s=55,
               edgecolor=SURFACE, linewidth=1.5, label=f"{prec.upper()} · forward", zorder=3)
    ax.scatter(d.e2e_median_ms, d.e2e_fps, facecolor="none", edgecolor=PREC[prec], marker=MARKER[prec],
               s=55, linewidth=1.5, label=f"{prec.upper()} · predict()", zorder=3)
    if prec == "fp32":
        for m, xv, yv in zip(MODELS, d.fwd_median_ms, d.fwd_fps):
            ax.annotate(SHORT[m], (xv, yv), xytext=(5, 4), textcoords="offset points", fontsize=8, color=INK_2)
ax.set_xlabel("median latency (ms, batch = 1)")
ax.set_ylabel("throughput (FPS)")
ax.grid(True, axis="both")
ax.legend(ncol=1, loc="upper right")
fig.tight_layout()
save(fig, "fig4_latency_vs_fps")
plt.show()
e.set_index(["model", "precision"])[
    ["fwd_median_ms", "fwd_p95_ms", "fwd_p99_ms", "fwd_fps", "e2e_median_ms", "e2e_p95_ms", "e2e_fps", "contended"]
].reindex(MODELS, level=0).round(2)

## Figure 5 — Latency distribution (median and P95)

Box = interquartile range, centre line = median, whiskers = P5–P95 of the timed forward passes
(warm-up excluded). A short upper whisker means a stable, predictable latency.

In [ ]:
raw = {}
for m in MODELS:
    for prec in ("fp32", "fp16"):
        p = PIPELINE_DIR / "phase5_test" / "efficiency" / f"{DEPLOY_VARIANT}_{m}_{prec}.json"
        if p.exists():
            raw[(m, prec)] = json.loads(p.read_text())["forward"]["raw_ms"]

fig, ax = plt.subplots(figsize=(7.2, 2.9))
width = 0.34
for i, prec in enumerate(("fp32", "fp16")):
    data = [raw.get((m, prec), [np.nan]) for m in MODELS]
    pos = X + (i - 0.5) * (width + 0.04)
    bp = ax.boxplot(data, positions=pos, widths=width, whis=(5, 95), showfliers=False, patch_artist=True,
                    medianprops=dict(color=SURFACE, linewidth=1.5),
                    boxprops=dict(facecolor=PREC[prec], edgecolor=PREC[prec]),
                    whiskerprops=dict(color=PREC[prec], linewidth=1.2), capprops=dict(color=PREC[prec], linewidth=1.2))
    bp["boxes"][0].set_label(prec.upper())
ax.set_xticks(X, [SHORT[m] for m in MODELS])
ax.set_xlabel("YOLO26-seg size")
ax.set_ylabel("forward latency (ms)")
ax.set_title(f"Forward-pass latency, batch = 1 ({DEPLOY_VARIANT})")
ax.legend(loc="upper left")
fig.tight_layout()
save(fig, "fig5_latency_distribution")
plt.show()
pd.DataFrame({f"{m}/{p}": {"median": np.median(v), "P95": np.percentile(v, 95), "P99": np.percentile(v, 99),
                           "P95 − median": np.percentile(v, 95) - np.median(v)} for (m, p), v in raw.items()}).T.round(3)

## Figure 6 — Memory footprint

Steady-state peak VRAM allocated by PyTorch during batch-1 inference (cuDNN autotuning workspaces of the
warm-up and the CUDA context are excluded), for the network forward pass and the full `predict()`,
and the peak host RAM (RSS) of the benchmark process.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.7))
panels = (("vram_peak_allocated_mb", "VRAM, forward (MB)"), ("vram_peak_allocated_e2e_mb", "VRAM, predict() (MB)"),
          ("ram_peak_rss_mb", "host RAM, peak RSS (MB)"))
w = 0.38
for ax, (col, title) in zip(axes, panels):
    for i, prec in enumerate(("fp32", "fp16")):
        d = e[e.precision == prec].set_index("model").reindex(MODELS)[col]
        ax.bar(X + (i - 0.5) * w, d, width=w, color=PREC[prec], edgecolor=SURFACE, linewidth=1.5, label=prec.upper())
    ax.set_xticks(X, [SHORT[m] for m in MODELS])
    ax.set_title(title, fontsize=9)
axes[0].legend(loc="upper left")
fig.tight_layout()
save(fig, "fig6_memory")
plt.show()
e.set_index(["model", "precision"])[
    ["vram_weights_mb", "vram_peak_allocated_mb", "vram_peak_allocated_e2e_mb", "ram_peak_rss_mb",
     "size_mb_disk", "size_mb_fp32_theoretical", "size_mb_fp16_theoretical", "params_fused", "gflops"]
].reindex(MODELS, level=0).round(1)

## Figure 7 — Accuracy–latency trade-off

Test DSC of the deployed variant against its end-to-end median latency, in FP32 and FP16
(FP16 accuracy is measured, not assumed). Models up and to the left dominate.

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.4))
for prec in ("fp32", "fp16"):
    a = acc[(acc.variant == DEPLOY_VARIANT) & (acc.precision == prec)].set_index("model").reindex(MODELS)
    l = e[e.precision == prec].set_index("model").reindex(MODELS)
    ax.plot(l.e2e_median_ms, a.dsc_mean, color=PREC[prec], marker=MARKER[prec], markeredgecolor=SURFACE,
            markeredgewidth=1.5, label=prec.upper())
    if prec == "fp32":
        for m, xv, yv in zip(MODELS, l.e2e_median_ms, a.dsc_mean):
            ax.annotate(SHORT[m], (xv, yv), xytext=(5, -10), textcoords="offset points", fontsize=8, color=INK_2)
ax.set_xlabel("end-to-end median latency (ms, batch = 1)")
ax.set_ylabel(f"DSC (test, {DEPLOY_VARIANT})")
ax.grid(True, axis="both")
ax.legend()
fig.tight_layout()
save(fig, "fig7_accuracy_latency_tradeoff")
plt.show()

<!-- standard-committee-figures -->
## Standard figure A — Accuracy vs. efficiency (deployed variant, batch = 1)

Test-set accuracy — overlap (DSC) and contour adherence (Boundary IoU) — against the three deployment costs the
committee asked for: median forward latency, throughput (FPS) and parameter count. Each point is one model at one
precision (FP32 circles, FP16 triangles); x-axes are logarithmic. Identical cell in the YOLO26, U-Net and SAM 3
notebooks; the root notebook `Article_Figures_and_Tables.ipynb` draws the same plot across architectures.

In [ ]:
d_acc = acc[acc.variant == DEPLOY_VARIANT]
d_eff = eff[eff.variant == DEPLOY_VARIANT]
both = d_acc.merge(d_eff, on=["variant", "model", "precision"], suffixes=("", "_eff"))
y_rows = [("dsc_mean", "DSC (test)")]
y_rows.append(("biou_mean", "Boundary IoU (test)") if "biou_mean" in both and both["biou_mean"].notna().any()
              else ("jsi_mean", "JSI (test)"))
x_cols = [("fwd_median_ms", "median forward latency (ms)"), ("fwd_fps", "throughput (FPS)"),
          ("params_fused", "parameters")]
fig, axes = plt.subplots(len(y_rows), len(x_cols), figsize=(7.2, 2.5 * len(y_rows)), squeeze=False)
for r, (yc, ylab) in enumerate(y_rows):
    for c, (xc, xlab) in enumerate(x_cols):
        ax = axes[r, c]
        for prec in ("fp32", "fp16"):
            d = both[both.precision == prec].set_index("model").reindex(MODELS)
            ax.scatter(d[xc], d[yc], color=PREC[prec], marker=MARKER[prec], s=45, edgecolor=SURFACE,
                       linewidth=1.5, label=prec.upper(), zorder=3)
            if prec == "fp32":
                for m, xv, yv in zip(MODELS, d[xc], d[yc]):
                    ax.annotate(SHORT[m], (xv, yv), xytext=(4, -9), textcoords="offset points", fontsize=7, color=INK_2)
        ax.set_xscale("log")
        ax.grid(True, axis="both")
        if r == len(y_rows) - 1:
            ax.set_xlabel(xlab)
        if c == 0:
            ax.set_ylabel(ylab)
axes[0, 0].legend(loc="lower left")
fig.tight_layout()
save(fig, "figA_accuracy_vs_efficiency")
plt.show()
both.set_index(["model", "precision"])[[c for c, _ in y_rows] + [c for c, _ in x_cols]].reindex(MODELS, level=0).round(4)

## Standard figure B — Training and inference time

Left: wall-clock training time per phase (from every run's `results.csv`, validation included; `summary/training_cost.csv`).
Right: batch-1 inference time of the deployed variant — network forward pass vs. the full prediction pipeline —
in FP32 and FP16.

In [ ]:
cost = read_csv("training_cost")
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8))
if len(cost):
    phases = ["phase1_baseline", "phase2_cv_baseline", "phase3_hpo", "phase4_optimized"]
    labels = {"phase1_baseline": "Ph1 baseline", "phase2_cv_baseline": "Ph2 CV (5 folds)",
              "phase3_hpo": "Ph3 HPO", "phase4_optimized": "Ph4 optimized"}
    colors = ["#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"]
    left = np.zeros(len(MODELS))
    for ph, col in zip(phases, colors):
        v = cost[cost.phase == ph].set_index("model").reindex(MODELS)["train_hours"].fillna(0).to_numpy()
        axes[0].barh(X, v, left=left, color=col, edgecolor=SURFACE, linewidth=1.5, label=labels[ph])
        left += v
    axes[0].set_yticks(X, [SHORT[m] for m in MODELS])
    axes[0].set_xlabel("training time (GPU hours)")
    axes[0].set_title("Training cost per phase")
    axes[0].grid(True, axis="x"); axes[0].grid(False, axis="y")
    axes[0].legend(fontsize=7, loc="lower right")
w = 0.2
for i, (prec, scope) in enumerate([(p, s) for p in ("fp32", "fp16") for s in ("fwd", "e2e")]):
    d = eff[(eff.variant == DEPLOY_VARIANT) & (eff.precision == prec)].set_index("model").reindex(MODELS)
    axes[1].bar(X + (i - 1.5) * w, d[f"{scope}_median_ms"], width=w, color=PREC[prec],
                alpha=1.0 if scope == "fwd" else 0.45, edgecolor=SURFACE, linewidth=1,
                label=f"{prec.upper()} {'forward' if scope == 'fwd' else 'end-to-end'}")
axes[1].set_xticks(X, [SHORT[m] for m in MODELS])
axes[1].set_yscale("log")
axes[1].set_ylabel("median latency (ms, log)")
axes[1].set_title("Inference time (batch = 1)")
axes[1].legend(fontsize=7)
fig.tight_layout()
save(fig, "figB_training_inference_time")
plt.show()
cost.pivot(index="model", columns="phase", values="train_hours").reindex(MODELS).round(2) if len(cost) else cost

## Standard figure C — Boundary metrics on the test set (Metrics Reloaded)

Boundary IoU (band = 2 % of the image diagonal) and Normalised Surface Distance (tolerance = 1 % of the diagonal),
per-image mean with seeded bootstrap 95 % CI, Baseline vs. Optimized (FP32). Overlap scores (DSC, JSI) are
dominated by the lesion interior; these two scores measure contour adherence. The paired Optimized − Baseline
differences (bootstrap CI, Wilcoxon *p*) are in `summary/hpo_gain.csv`.

In [ ]:
if "biou_mean" not in acc32 or acc32["biou_mean"].isna().all():
    print("Boundary metrics not in test_accuracy.csv — re-run Phase 5a (evaluate_test_set.py, EVAL_VERSION 2).")
else:
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.7))
    for ax, k, title in zip(axes, ("biou", "nsd"), ("Boundary IoU", "Normalised Surface Distance")):
        for off, v in ((-0.12, "baseline"), (0.12, "optimized")):
            d = acc32[acc32.variant == v].set_index("model").reindex(MODELS)
            m_ = d[f"{k}_mean"].to_numpy()
            err = np.vstack([m_ - d[f"{k}_ci95_low"].to_numpy(), d[f"{k}_ci95_high"].to_numpy() - m_])
            ax.errorbar(X + off, m_, yerr=err, fmt=MARKER[v], color=SERIES[v], elinewidth=1.2, capsize=0,
                        markeredgecolor=SURFACE, markeredgewidth=1.5, markersize=7, label=v.capitalize())
        ax.set_xticks(X, [SHORT[m] for m in MODELS])
        ax.set_xlim(-0.6, len(MODELS) - 0.4)
        ax.set_title(title)
    axes[0].set_ylabel("per-image mean (95 % CI)")
    axes[0].legend(loc="lower left")
    fig.tight_layout()
    save(fig, "figC_boundary_metrics")
    plt.show()
    display(acc32.set_index(["model", "variant"])[[c for c in acc32.columns if c.startswith(("biou_", "nsd_"))]]
            .reindex(MODELS, level=0).round(4))

## LaTeX tables
`booktabs` tables for the thesis, written to `<pipeline>/tables/`.

In [ ]:
def to_latex(df: pd.DataFrame, name: str, caption: str) -> None:
    try:
        tex = df.to_latex(float_format="%.4f", caption=caption, label=f"tab:{name}", escape=True)
    except ImportError as err:  # pandas' LaTeX writer needs jinja2 (bundled with jupyterlab)
        print(f"[skip] {name}: {err}")
        return
    (TABLE_DIR / f"{name}.tex").write_text(tex)
    print(f"saved {TABLE_DIR / name}.tex")


if len(acc32):
    t = acc32.pivot(index="model", columns="variant",
                    values=[k for k in ("dsc_mean", "jsi_mean", "jsi_thr_mean", "biou_mean", "nsd_mean", "map5095_m") if k in acc32]).reindex(MODELS)
    t.columns = [f"{v} {k.replace('_mean', '').upper()}" for k, v in t.columns]
    to_latex(t, "test_accuracy", "Test-set accuracy of the Baseline and Optimized YOLO26-seg models (FP32).")
if len(e):
    t = e.set_index(["model", "precision"])[
        ["params_fused", "gflops", "size_mb_disk", "fwd_median_ms", "fwd_p95_ms", "fwd_fps", "vram_peak_allocated_mb"]
    ].reindex(MODELS, level=0)
    to_latex(t, "efficiency", "Batch-1 efficiency of the deployed YOLO26-seg models (FP32 vs. FP16).")

## Reading notes

* **Contended runs.** `contended = True` means another process used the GPU during the benchmark;
  re-run Phase 5b on an idle GPU before quoting those latencies.
* **Model size.** Ultralytics stores final checkpoints in FP16, so `size_mb_disk` ≈ the FP16 size; the
  theoretical FP32/FP16 sizes are fused parameters × 4 / × 2 bytes.
* **VRAM.** PyTorch-allocator figures at steady state; the CUDA context (constant, a few hundred MB) is excluded.
* **CV vs. test.** CV uses the Baseline protocol on held-out folds of train+val; only Phase 5 touches the test set.